## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [11]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np

np.random.seed(50)  # random seed for reproducability

In [12]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [13]:
# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

eval_cal_indices = np.concatenate([eval_indices, calibration_indices])
eval_cal_indices_sorted = np.sort(eval_cal_indices)

In [14]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :N_train

n_train = len(train_indices)  # number of training samples

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

business_covariates

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,...,chain,density,TRAIN,category,FT,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre
0,QkG3KUXwqZBW18A9k1xqCA,"""Red Lobster""",NaN,"""2810 North 75th Ave""",Phoenix,AZ,85035.0,33.478735,-112.221379,2.5,...,1,11,1,American,False,2.0,700.0,350.0,1248.0,14035.314000
1,mGRUkg7YO1fWpfYCqNcR-g,"""The Corner CityScape""",NaN,"""50 W Jefferson St""",Phoenix,AZ,85004.0,33.447175,-112.074834,3.5,...,0,56,1,American,False,2.0,139.0,58.0,1495.0,477.950899
2,vUgCHg2pWUrbg_xrJZz6Lg,"""Desert Donuts""",NaN,"""3134 W Carefree Hwy, Ste A-10""",Phoenix,AZ,85086.0,33.799869,-112.127647,4.5,...,0,11,1,Cafes,False,1.0,50.0,15.0,1730.0,39112.010152
3,VYuMUCoN6LWTxM80_itq1Q,"""Wolfley's Neighborhood Grill""",NaN,"""21001 N Tatum Blvd, Ste 96""",Phoenix,AZ,85050.0,33.677102,-111.975283,3.5,...,0,15,1,American,False,2.0,NaN,NaN,1638.0,26737.847111
4,U5Ow3ffmrG4MmTZLK6jd-g,"""Soup & Sausage Bistro""",NaN,"""13240 N 7th St""",Phoenix,AZ,85022.0,33.606836,-112.065705,4.5,...,0,10,1,Other,False,2.0,60.0,30.0,1431.0,17320.797326
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,T45K4FeSLlOX_TjEvMImzg,"""Scott's Generations Restaurant & Delicatessen""",NaN,"""742 E Glendale Ave, Ste 142""",Phoenix,AZ,85020.0,33.538930,-112.063828,4.0,...,0,3,0,Cafes,False,2.0,243.0,142.0,1515.0,9789.331620
917,3-aEgS7X2jrbxA7sA1nARw,"""La Flor De Calabaza""",NaN,"""705 N 1st St, Ste 110""",Phoenix,AZ,85004.0,33.455863,-112.072167,3.0,...,0,62,0,Mexican,False,2.0,303.0,198.0,1495.0,521.853595
918,3sOS8wKatd_Uoa9VEJHhrw,"""Miracle Mile Delicatessen""",NaN,"""4433 N 16th St""",Phoenix,AZ,85016.0,33.501659,-112.047250,4.0,...,0,8,0,American,False,2.0,643.0,444.0,1710.0,6116.608365
919,wAXYLmHuysYTz8i4VPKmaQ,"""Chipotle Mexican Grill""",NaN,"""425 E Bell Rd, Ste 140""",Phoenix,AZ,85022.0,33.639893,-112.067625,3.5,...,1,19,0,Mexican,False,1.0,166.0,68.0,1431.0,20992.160279


In [15]:
# initialize new lists

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")

[0] - Conversion for business_id: QkG3KUXwqZBW18A9k1xqCA
[100] - Conversion for business_id: CR-gLUcudD0AQr7dzASgLA
[200] - Conversion for business_id: lpQziF9QCVZQRkxac1xzcw
[300] - Conversion for business_id: XF4w6wG1JOvfKZlQcxiWgQ
[400] - Conversion for business_id: 7S5iOI5Xb9cTUQLmBwCeXw
[500] - Conversion for business_id: sYmGrCydqABvGmundtZ6gg
[600] - Conversion for business_id: 2GryItAj3e0uOQ5m42-wDg
[700] - Conversion for business_id: BG_l5Fp-aBOAxudpl6wStQ
[800] - Conversion for business_id: iiuEt92eGGy_fTvywsTBbA
[900] - Conversion for business_id: tcPVY5QCFL_roeouCY5SXQ
Done ... validation checks passed!


In [16]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# R: mutate(l_age = log(Age), Checkin = Checkin/Age*28)
# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis (same as R: l_age)
business_covariates["logAge"] = np.log(business_covariates["Age"])

# R: Covariates <- covariates_business %>%
#    select(density, Checkin, category, chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age)
# only get relevant covariates for analysis (must match R order and selection)
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,American,1,2.0,700.0,350.0,1248.0,2057
1,56,15.482866,American,0,2.0,139.0,58.0,1495.0,642
2,11,3.164141,Cafes,0,1.0,50.0,15.0,1730.0,1584
3,15,14.023769,American,0,2.0,NaN,NaN,1638.0,1767
4,10,4.153110,Other,0,2.0,60.0,30.0,1431.0,209
...,...,...,...,...,...,...,...,...,...
916,3,8.230032,Cafes,0,2.0,243.0,142.0,1515.0,313
917,62,2.402307,Mexican,0,2.0,303.0,198.0,1495.0,1387
918,8,34.600877,American,0,2.0,643.0,444.0,1710.0,912
919,19,7.975097,Mexican,1,1.0,166.0,68.0,1431.0,2570


In [17]:
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding and Scaling  already performed on dataframe!"

# R: options(na.action="na.pass")
# R: cov_mat <- model.matrix(formula(paste("~",paste(names(Covariates),collapse = "+"),"-1")),
#                             data = Covariates)[,-8]

# The formula in R is: ~ density + Checkin + category + chain + Price.Level +
#                        Restaurant.Size + Number.of.Seats + ZRI + Age - 1
# model.matrix creates columns in this order:
# density, Checkin, categoryAmerican, categoryAsian, categoryCafes, categoryFast Food,
# categoryMexican, categoryOther, categoryPizza, categorySalad, categorySpeciality Food,
# chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age
# Then [,-8] removes column 8 which is "categoryOther"

# Convert category to factor (like R)
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# Create dummy variables - R's model.matrix with "-1" creates all categories (no baseline)
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

# R model.matrix order: numeric columns in original order, then categorical dummies alphabetically
# Original order: density, Checkin, category (becomes multiple), chain, Price.Level,
#                 Restaurant.Size, Number.of.Seats, ZRI, Age

# First two numeric columns before category
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# Combine in R's order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# R: [,-8] removes column 8 (1-indexed in R)
# This is column index 7 in Python (0-indexed)
# Based on the order above, column 8 is "categoryOther"
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (R column 8): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")
        # Still remove it to match R behavior
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (R column 8): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057


In [18]:
# R: preProc <- preProcess(cov_mat[1:500,], c("center","medianImpute"))
# Python equivalent: First fit on training data, then center, then impute
# Note: R's caret::preProcess with c("center", "medianImpute") first centers, then imputes with median

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler(with_std=False)  # only centering, no scaling!


X_train = relevant_covariates.iloc[:n_train].copy()

# R applies: preProcess(cov_mat[1:500,], c("center","medianImpute"))
# This means: calculate center from training data, then impute missing values with median
# In R's caret, the order in the vector matters - "center" is applied first to calculate statistics
# but "medianImpute" fills NAs before centering is applied in the transform step

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition (same as R: qr() function)
Q, R = np.linalg.qr(X_train_preprocessed)

# R: Q <- qr.Q(QR)*sqrt(N_train-1)
# R: R <- qr.R(QR)/sqrt(N_train-1)
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

In [19]:
from helpers import comp_entropy

# aggregate review stats
# R column names: VAR, MEAN, ENTR, COUNT, ONE_STAR, TWO_STAR, THREE_STAR, FOUR_STAR, FIVE_STAR
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities (same as R)
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# R: select(business_id, density, Checkin, category, chain, Price.Level,
#           Restaurant.Size, Number.of.Seats, ZRI, Distance.To.City.Centre, Age, is_open)
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# R: mutate(Closed = 1-is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# R: left_join(temp, by="business_id")
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# R: mutate(l_COUNT = log(COUNT), category = factor(category), Closed = factor(Closed))
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical (same as R factors)
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# R: Closed <- fct_recode(Closed, "Closed" = "1", "Open" = "0")
# Convert Closed to categorical with proper labels
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

Benchmark covariates prepared with shape: (921, 23)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,VAR,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,l_COUNT
0,QkG3KUXwqZBW18A9k1xqCA,11,2.327662,American,1,2.0,700.0,350.0,1248.0,14035.314000,...,2.900901,2.648649,1.421063,37,0.432432,0.108108,0.081081,0.135135,0.243243,3.610918
1,mGRUkg7YO1fWpfYCqNcR-g,56,15.482866,American,0,2.0,139.0,58.0,1495.0,477.950899,...,1.963646,3.492063,1.521366,63,0.126984,0.158730,0.111111,0.301587,0.301587,4.143135
2,vUgCHg2pWUrbg_xrJZz6Lg,11,3.164141,Cafes,0,1.0,50.0,15.0,1730.0,39112.010152,...,0.616880,4.691429,0.650410,175,0.011429,0.040000,0.011429,0.120000,0.817143,5.164786
3,VYuMUCoN6LWTxM80_itq1Q,15,14.023769,American,0,2.0,NaN,NaN,1638.0,26737.847111,...,1.716087,3.327869,1.557455,183,0.131148,0.142077,0.202186,0.316940,0.207650,5.209486
4,U5Ow3ffmrG4MmTZLK6jd-g,10,4.153110,Other,0,2.0,60.0,30.0,1431.0,17320.797326,...,0.706736,4.690476,0.642852,42,0.023810,0.023810,0.023810,0.095238,0.833333,3.737670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,T45K4FeSLlOX_TjEvMImzg,3,8.230032,Cafes,0,2.0,243.0,142.0,1515.0,9789.331620,...,2.461202,3.852459,1.247320,61,0.163934,0.081967,0.065574,0.114754,0.573770,4.110874
917,3-aEgS7X2jrbxA7sA1nARw,62,2.402307,Mexican,0,2.0,303.0,198.0,1495.0,521.853595,...,3.114237,2.942029,1.380396,69,0.391304,0.057971,0.101449,0.115942,0.333333,4.234107
918,3sOS8wKatd_Uoa9VEJHhrw,8,34.600877,American,0,2.0,643.0,444.0,1710.0,6116.608365,...,1.311242,4.235955,1.167207,178,0.056180,0.044944,0.089888,0.224719,0.584270,5.181784
919,wAXYLmHuysYTz8i4VPKmaQ,19,7.975097,Mexican,1,1.0,166.0,68.0,1431.0,20992.160279,...,2.654716,3.253012,1.465422,83,0.277108,0.072289,0.084337,0.253012,0.313253,4.418841


In [20]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=np.arange(500),
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / "processed_data.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...

